[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/05_your_first_real_network.ipynb)

# ⚒️ Act I · Quest 05 — Your First Real Network

*nn.Module, optimizers, losses — the industrial version of the MLP you hand-forged.*

⬅️ [04_autograd_unmasked](04_autograd_unmasked.ipynb)  •  [06_feeding_the_beast](06_feeding_the_beast.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## From your `MLP` class to `nn.Module`

In Quest 02 you wrote `Neuron`, `Layer`, `MLP` and tracked parameters by hand. PyTorch's
`nn.Module` is that pattern, productionized: it auto-registers parameters, moves them between
devices, saves/loads them, and composes into arbitrarily deep trees of modules.

The mapping is exact:

| your forge | PyTorch |
|------------|---------|
| `Neuron` / `Layer` | `nn.Linear` |
| `.tanh()` in the neuron | separate activation (`nn.Tanh`, `nn.ReLU`) |
| `net.parameters()` list | `model.parameters()` generator |
| `p.data -= lr * p.grad` loop | `optimizer.step()` |
| `p.grad = 0` loop | `optimizer.zero_grad()` |
| `sum((p-y)**2)` | `nn.MSELoss`, `nn.CrossEntropyLoss`, ... |

### The arena: two interlocking spirals (much nastier than XOR)

In [ ]:
def make_spirals(n=600, noise=0.12, turns=3):
    n2 = n // 2
    idx = torch.arange(n2).float()
    r = idx / n2 * 1.6
    th = idx / n2 * turns * torch.pi
    s0 = torch.stack([r * torch.sin(th), r * torch.cos(th)], dim=1)
    s1 = torch.stack([r * torch.sin(th + torch.pi), r * torch.cos(th + torch.pi)], dim=1)
    X = torch.cat([s0, s1]) + noise * torch.randn(n, 2)
    y = torch.cat([torch.zeros(n2), torch.ones(n2)]).long()
    perm = torch.randperm(n)
    return X[perm], y[perm]

X, y = make_spirals()
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10)
plt.title("Two spirals — try drawing a straight line through THAT"); plt.axis("equal"); plt.show()

### Define the model

In [ ]:
class SpiralNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2),          # 2 logits: one score per class
        )
    def forward(self, x):
        return self.body(x)

model = SpiralNet().to(device)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

### The five-line liturgy

You will write this loop hundreds of times in your life. Learn it as a rhythm:

```
zero → forward → loss → backward → step
```

In [ ]:
X, y = X.to(device), y.to(device)
criterion = nn.CrossEntropyLoss()                       # softmax + NLL in one, expects raw logits
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

losses = []
for epoch in range(600):
    optimizer.zero_grad()          # 1. zero   (Rule 1 of autograd!)
    logits = model(X)              # 2. forward
    loss = criterion(logits, y)    # 3. loss
    loss.backward()                # 4. backward
    optimizer.step()               # 5. step
    losses.append(loss.item())

acc = (model(X).argmax(1) == y).float().mean().item()
print(f"loss {losses[-1]:.4f} | accuracy {acc*100:.1f}%")
plt.plot(losses); plt.title("The liturgy, working"); plt.xlabel("epoch"); plt.show()

In [ ]:
# The decision boundary — compare with what a straight line could ever do
def plot_boundary(model, X, y):
    xs = torch.linspace(X[:, 0].min() - .3, X[:, 0].max() + .3, 250)
    ys = torch.linspace(X[:, 1].min() - .3, X[:, 1].max() + .3, 250)
    XX, YY = torch.meshgrid(xs, ys, indexing="xy")
    grid = torch.stack([XX.flatten(), YY.flatten()], dim=1).to(device)
    with torch.no_grad():
        Z = model(grid).argmax(1).reshape(XX.shape).cpu()
    plt.contourf(XX, YY, Z, alpha=0.3, cmap="coolwarm")
    plt.scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=y.cpu(), cmap="coolwarm", s=8, edgecolors="k", linewidths=0.2)
    plt.axis("equal")

plot_boundary(model, X, y)
plt.title("A learned spiral boundary — this is why depth matters"); plt.show()

### Optimizers: smarter ways to roll downhill

Plain SGD steps straight down the local slope. **Momentum** remembers velocity; **Adam** adapts
a per-knob learning rate. Same valley, different descent styles:

In [ ]:
results = {}
for name, make_opt in {
    "SGD":      lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD+mom":  lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam":     lambda p: torch.optim.Adam(p, lr=0.01),
}.items():
    torch.manual_seed(7)
    m = SpiralNet().to(device)
    opt = make_opt(m.parameters())
    hist = []
    for _ in range(300):
        opt.zero_grad()
        l = criterion(m(X), y)
        l.backward(); opt.step()
        hist.append(l.item())
    results[name] = hist

for name, hist in results.items():
    plt.plot(hist, label=name)
plt.legend(); plt.title("Same net, three optimizers"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.show()

### `train()` / `eval()` and saving your work

Two module modes: `model.train()` (dropout on, batchnorm updating) and `model.eval()`
(deterministic inference). Pair `eval()` with `no_grad()`. And save the `state_dict` — just the
weights, not the code.

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/spiralnet.pt")

revived = SpiralNet().to(device)
revived.load_state_dict(torch.load("checkpoints/spiralnet.pt", map_location=device))
revived.eval()
with torch.no_grad():
    same = torch.equal(revived(X).argmax(1), model(X).argmax(1))
print("revived model agrees with original:", same)

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda n: n == sum(p.numel() for p in nn.Linear(10, 5).parameters()),
    "nn.Linear(10, 5) has 10*5 weights + 5 biases = 55")
_register("liturgy", 15,
    lambda steps: [s.strip().lower() for s in steps] == ["zero", "forward", "loss", "backward", "step"],
    "submit the 5 steps of the training loop as a list of lowercase words")
_register("spiral_95", 20,
    lambda m: isinstance(m, nn.Module) and (m(X).argmax(1) == y).float().mean().item() >= 0.95,
    "train a model (any architecture) to >= 95% accuracy on the spirals X, y and submit the model")
_register("custom_block", 15,
    lambda m: isinstance(m, nn.Module) and m(torch.randn(3, 8)).shape == (3, 8)
              and any(p.numel() for p in m.parameters()),
    "an nn.Module whose forward returns x + linear(x).relu() — a residual block! in: (B,8), out: (B,8)")

In [ ]:
check("warmup", 55)

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `liturgy` (15 XP) — the five steps, in order, as a list of words.
- `spiral_95` (20 XP) — train any model to ≥95% on the spirals; submit the trained model.
- `custom_block` (15 XP) — write a **residual block** module: `forward(x) = x + relu(linear(x))` for 8-dim inputs; submit an instance.

In [ ]:
# ⚔️ your attempts here...

# xp_report()

---
## 🏁 End of Act I

Take stock of what you can now do: derive learning from scratch, **build an autograd engine**,
wield tensors, and train real networks with the five-line liturgy. Act II puts specialized
senses on your networks: eyes (convolutions), memory (recurrence), and attention (GPT).